# `basic_cells_B.ipynb` — Path B (AutoGluon Chronos-2 fine-tuning) cell library

Part B counterpart of `basic_cells_A.ipynb`. Single source of truth for Path B code;
sourced by `runner_B.ipynb` via `%run basic_cells_B.ipynb` (same idiom as Part A).

path_b_cells.py — Path B (AutoGluon Chronos-2 fine-tuning) cell library for MOEX.

This module is the single source of truth for Path B code, mirroring the role
`basic_cells.ipynb` plays for Path A. It is imported by `runner_path_b.ipynb`.

WHY A SEPARATE MODULE (deviation from path_b_implementation_prompt.md step A):
  The implementation prompt asks to "extend basic_cells.ipynb". The user's final
  instruction was "Implement within path_b folder only. Let the rest be unchanged."
  Those conflict. We honor the user's later, explicit constraint: `basic_cells.ipynb`
  is left UNTOUCHED, and the Path A experiment logic it contains (panel build,
  walk-forward anchors, per-cell metrics, baselines, plots, Chronos input builder)
  is PORTED verbatim into Section 0 below — adapted only where Path B differs
  (horizons [1,2,3], prediction_length=3). The Path A *data-loading* side
  (ISS prefetch, FORTS resolver, cache loaders) is NOT ported here; the runner
  loads it at runtime via `%run ../basic_cells.ipynb` (it needs Colab + Drive).
  This decision is recorded in path_b/current_state.md.

## 0. Imports

In [ ]:
import os
import json
import time
import copy
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd

try:
    import yaml
except Exception:  # pragma: no cover - yaml always present in the project env
    yaml = None

try:
    import scipy.stats as sstats
except Exception:  # pragma: no cover
    sstats = None

# Matplotlib is import-guarded so the module imports even in a headless/no-mpl env;
# plotting functions raise a clear error if called without it.
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover
    plt = None

# tqdm is required by the prompt for Path B loops; fall back to a no-op if missing.
try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

## 1. Section 0 — Ported Path A logic (from basic_cells.ipynb; basic_cells.ipynb UNTOUCHED)

In [ ]:
# These functions are copied faithfully from Path A. Where Path B differs from Path A
# the difference is ONLY in config defaults (prediction_length=3, horizons [1,2,3]),
# which is handled in Section A (normalize_path_b_config) — the functions below are
# horizon-agnostic and reused unchanged.

# --- 0.1 Config loader (ported from basic_cells §2) ----------------------------------
PATH_A_REQUIRED_KEYS = [
    "stage_id", "interval", "tickers", "date_from", "date_till",
    "context_len", "horizon", "walk_forward", "covariates", "output_dir",
]


def load_config(path: str) -> dict:
    """Load + validate a stage YAML (Path A schema). Ported from basic_cells.ipynb §2."""
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    missing = [k for k in PATH_A_REQUIRED_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"config {path} missing required keys: {missing}")
    assert cfg["context_len"] <= 8192, "Chronos-2 max context is 8192"
    assert cfg["horizon"] <= 1024, "Chronos-2 max prediction_length is 1024"
    assert cfg["interval"] in (10, 60, 24), "interval must be 10, 60, or 24 (ISS has no 15m candle)"
    cfg.setdefault("indexes", ["IMOEX", "MOEXOG", "MOEXMM", "MOEXFN", "RGBI"])
    cfg.setdefault("futures_proxies", [])
    cfg.setdefault("quantiles", [0.1, 0.5, 0.9])
    # Path A defaults; Path B overrides via normalize_path_b_config (reads cfg["path_b"][...])
    cfg.setdefault("eval_horizons", [1, 2, 3, 5])
    cfg.setdefault("primary_horizons", [2, 3, 5])
    cfg.setdefault("baselines", ["zero", "last", "momentum5", "ar1"])
    return cfg


def save_config_snapshot(cfg: dict, out_dir: str):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(out_dir) / "config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)


# --- 0.2 Panel build + covariates + calendar (ported from basic_cells §6) ------------
def _session_grid(start, end, interval_min):
    days = pd.bdate_range(start.normalize(), end.normalize())
    bars_per_day = (8 * 60 + 50) // interval_min  # 10:00..18:50 MSK
    out = [pd.date_range(d + pd.Timedelta(hours=10), periods=bars_per_day, freq=f"{interval_min}min") for d in days]
    return pd.DatetimeIndex(np.concatenate(out)) if out else pd.DatetimeIndex([])


def to_regular_series(df, value_col, name, interval):
    if df.empty:
        return pd.Series(name=name, dtype=float)
    s = df.set_index("begin")[value_col].astype(float).sort_index()
    if interval == 24:
        s = s.asfreq("B")
    else:
        s = s.reindex(_session_grid(s.index.min(), s.index.max(), interval))
    return s.ffill().rename(name)


def build_price_panel(prices, interval):
    return pd.concat([to_regular_series(df, "close", t, interval) for t, df in prices.items()], axis=1).dropna(how="any")


def log_returns(panel):
    return np.log(panel / panel.shift(1)).dropna(how="any")


def calendar_features(index, interval):
    df = pd.DataFrame({
        "hour":  index.hour.astype(np.float32),
        "dow":   index.dayofweek.astype(np.float32),
        "dom":   index.day.astype(np.float32),
        "month": index.month.astype(np.float32),
    }, index=index)
    if interval != 24:
        df["session_open"] = ((index.hour >= 10) & (index.hour < 19)).astype(np.float32)
    return df


# Calendar feature column names — the canonical "known-future" covariate set (leakage-safe).
def calendar_feature_names(interval):
    base = ["hour", "dow", "dom", "month"]
    if interval != 24:
        base = base + ["session_open"]
    return base


def build_covariate_panel(prices, indexes, futures, interval, ret_index):
    parts = []
    for name, df in indexes.items():
        s = to_regular_series(df, "close", f"{name}_close", interval)
        parts.append(np.log(s / s.shift(1)).rename(f"{name}_ret"))
    for name, df in futures.items():
        s_close = to_regular_series(df, "close", f"{name}_close", interval)
        cid_src = df.set_index("begin")["contract_id"]
        s_cid = cid_src.reindex(s_close.index, method="ffill")
        ret = np.log(s_close / s_close.shift(1))
        boundary = (s_cid != s_cid.shift(1)).fillna(False)
        ret = ret.mask(boundary)
        parts.append(ret.rename(f"{name}_ret"))
    for tic, df in prices.items():
        v = to_regular_series(df, "volume", f"{tic}_vol", interval)
        parts.append(np.log1p(v).diff().rename(f"{tic}_dlogvol"))
    cov = pd.concat(parts, axis=1).reindex(ret_index).ffill().dropna(how="any")
    return cov


def assemble_panels(cfg, prices, indexes, futures):
    price = build_price_panel(prices, cfg["interval"])
    ret = log_returns(price)
    cov_mode = cfg["covariates"]  # "full", "calendar_only", "market_only", "none"
    if cov_mode == "none":
        cov = pd.DataFrame(index=ret.index)
    else:
        cov = build_covariate_panel(prices, indexes, futures, cfg["interval"], ret.index)
        if cov_mode == "calendar_only":
            cov = cov.iloc[:, 0:0]
    common = ret.index.intersection(cov.index) if not cov.empty else ret.index
    return price.loc[common], ret.loc[common], cov.loc[common]


# --- 0.3 Chronos input builder (ported from basic_cells §7) --------------------------
def build_chronos_inputs(ret_panel, cov_panel, tickers, context_len, horizon, t_anchor,
                         interval, covariate_mode="full"):
    """t_anchor = first index of the forecast horizon (context ends at t_anchor-1).
    Returns (context_df, future_df, fut_idx). Ported from basic_cells.ipynb §7."""
    pos = ret_panel.index.get_loc(t_anchor)
    ctx_idx = ret_panel.index[pos - context_len:pos]
    fut_idx = ret_panel.index[pos:pos + horizon]

    use_market_cov = covariate_mode in ("full", "market_only") and not cov_panel.empty
    use_calendar = covariate_mode in ("full", "calendar_only")
    cal_ctx = calendar_features(ctx_idx, interval) if use_calendar else None
    cal_fut = calendar_features(fut_idx, interval) if use_calendar else None
    cov_ctx = cov_panel.loc[ctx_idx] if use_market_cov else None

    ctx_rows, fut_rows = [], []
    for tic in tickers:
        block = pd.DataFrame({"id": tic, "timestamp": ctx_idx, "target": ret_panel[tic].loc[ctx_idx].values})
        if use_market_cov:
            for c in cov_ctx.columns:
                block[c] = cov_ctx[c].values
        if use_calendar:
            for c in cal_ctx.columns:
                block[c] = cal_ctx[c].values
        ctx_rows.append(block)
        fb = pd.DataFrame({"id": tic, "timestamp": fut_idx})
        if use_calendar:
            for c in cal_fut.columns:
                fb[c] = cal_fut[c].values
        fut_rows.append(fb)
    return pd.concat(ctx_rows, ignore_index=True), pd.concat(fut_rows, ignore_index=True), fut_idx


# --- 0.4 Walk-forward anchors (ported from basic_cells §8) ---------------------------
def walk_forward_anchors(ret_index, context_len, horizon, shift, max_windows=None):
    starts = []
    pos = context_len
    end_pos = len(ret_index) - horizon
    while pos < end_pos:
        starts.append(ret_index[pos])
        pos += shift
    if max_windows and len(starts) > max_windows:
        idx = np.linspace(0, len(starts) - 1, max_windows).astype(int)
        starts = [starts[i] for i in idx]
    return starts


# --- 0.5 Metrics (ported from basic_cells §9) ----------------------------------------
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (center - half, center + half)


def per_cell_metrics(pred_df, ret_panel, tickers, horizons, quantiles):
    """Per (ticker, horizon-step): DA + binomial p + Wilson CI, Pearson, Spearman,
    |pred|/|true|, q-coverage, MAE, BH-adjusted q. Ported from basic_cells.ipynb §9."""
    if pred_df.empty or "id" not in pred_df.columns:
        return pd.DataFrame()
    median_q = str(quantiles[len(quantiles) // 2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].copy()
        if sub.empty:
            continue
        sub = sub.sort_values(["window", "timestamp"])
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        for h in horizons:
            slab = sub[sub["h_step"] == h]
            if slab.empty:
                continue
            y_pred = slab[median_q].values
            ts = slab["timestamp"].values
            try:
                y_true = ret_panel[tic].loc[pd.DatetimeIndex(ts)].values
            except KeyError:
                continue
            mask = ~(np.isnan(y_pred) | np.isnan(y_true))
            y_pred, y_true = y_pred[mask], y_true[mask]
            n = len(y_true)
            if n < 5:
                continue
            hits = int((np.sign(y_pred) == np.sign(y_true)).sum())
            da = hits / n
            p_binom = sstats.binomtest(hits, n, 0.5, alternative="greater").pvalue
            ci_lo, ci_hi = wilson_ci(hits, n)
            const_input = n < 2 or np.std(y_pred) == 0 or np.std(y_true) == 0
            r_p = np.nan if const_input else float(np.corrcoef(y_pred, y_true)[0, 1])
            r_s = np.nan if const_input else float(sstats.spearmanr(y_pred, y_true).correlation)
            amp = float(np.median(np.abs(y_pred) / (np.abs(y_true) + 1e-12)))
            q_lo = slab[lo_q].values[mask]
            q_hi = slab[hi_q].values[mask]
            cov = float(((y_true >= q_lo) & (y_true <= q_hi)).mean())
            mae = float(np.mean(np.abs(y_true - y_pred)))
            rows.append(dict(ticker=tic, horizon=h, n=n, da=da, p_binom=p_binom,
                             da_ci_lo=ci_lo, da_ci_hi=ci_hi, pearson=r_p, spearman=r_s,
                             amp_ratio=amp, coverage=cov, mae=mae))
    out = pd.DataFrame(rows)
    if not out.empty:
        m = len(out)
        order = out["p_binom"].rank(method="first").astype(int).values
        out["p_binom_bh"] = np.minimum.accumulate(
            (out["p_binom"].sort_values().values * m / np.arange(1, m + 1))[::-1]
        )[::-1][order - 1]
    return out


def aggregate_metrics(per_cell):
    if per_cell.empty:
        return pd.DataFrame()
    agg = per_cell.groupby("horizon").agg(
        mean_da=("da", "mean"), median_da=("da", "median"),
        mean_pearson=("pearson", "mean"), mean_spearman=("spearman", "mean"),
        mean_amp=("amp_ratio", "mean"), mean_cov=("coverage", "mean"),
        n_cells=("ticker", "count"),
        n_signif_05=("p_binom_bh", lambda s: int((s < 0.05).sum())),
    ).reset_index()
    return agg


# --- 0.6 Baselines (ported from basic_cells §10) -------------------------------------
def baseline_predictions(name, ret_panel, tickers, anchors, context_len, horizon):
    rows = []
    for i, t in enumerate(anchors):
        pos = ret_panel.index.get_loc(t)
        for tic in tickers:
            ctx = ret_panel[tic].iloc[pos - context_len:pos].values
            fut_ts = ret_panel.index[pos:pos + horizon]
            if name == "zero":
                yhat = np.zeros(horizon)
            elif name == "last":
                yhat = np.full(horizon, ctx[-1] if len(ctx) else 0.0)
            elif name == "momentum5":
                yhat = np.full(horizon, np.nanmean(ctx[-5:]) if len(ctx) >= 5 else 0.0)
            elif name == "ar1":
                if len(ctx) >= 30:
                    x, y = ctx[:-1], ctx[1:]
                    a = np.dot(x, y) / (np.dot(x, x) + 1e-12)
                    yhat = np.empty(horizon)
                    last = ctx[-1]
                    for k in range(horizon):
                        last = a * last
                        yhat[k] = last
                else:
                    yhat = np.zeros(horizon)
            else:
                raise ValueError(name)
            for k, ts in enumerate(fut_ts):
                rows.append(dict(id=tic, timestamp=ts, window=i, t_anchor=t,
                                 **{"0.5": yhat[k], "0.1": yhat[k], "0.9": yhat[k]}))
    return pd.DataFrame(rows)


# --- 0.7 Path A plots (ported from basic_cells §11) ----------------------------------
def _require_mpl():
    if plt is None:
        raise RuntimeError("matplotlib not available; install matplotlib to render plots")


def plot_da_heatmap(per_cell, out_path, title=""):
    _require_mpl()
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="da")
    fig, ax = plt.subplots(figsize=(1.2 * len(pivot.columns) + 2, 0.4 * len(pivot.index) + 2))
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.40, vmax=0.60, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="Dir. Acc.")
    ax.set_title(title or "Directional accuracy (ticker × horizon)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def plot_corr_hist(pred_df, ret_panel, horizons, quantiles, out_path):
    _require_mpl()
    median_q = str(quantiles[len(quantiles) // 2])
    fig, axes = plt.subplots(1, len(horizons), figsize=(4 * len(horizons), 3), sharey=True)
    if len(horizons) == 1:
        axes = [axes]
    pred_df = pred_df.sort_values(["window", "id", "timestamp"])
    pred_df["h_step"] = pred_df.groupby(["window", "id"]).cumcount() + 1
    for ax, h in zip(axes, horizons):
        rs = []
        slab = pred_df[pred_df["h_step"] == h]
        for (w, _id), s in slab.groupby(["window", "id"]):
            y_p = s[median_q].values
            try:
                y_t = ret_panel[_id].loc[s["timestamp"].values].values
            except KeyError:
                continue
            if len(y_t) > 1 and not np.isnan(y_p).any():
                rs.append(np.corrcoef(y_p, y_t)[0, 1])
        ax.hist(rs, bins=30)
        ax.axvline(0, color="red", lw=0.8)
        ax.set_title(f"h={h}, n={len(rs)}")
        ax.set_xlabel("Pearson r")
    plt.suptitle("Per-window correlation (pred vs true), by horizon")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def plot_amplitude(per_cell, out_path):
    _require_mpl()
    fig, ax = plt.subplots(figsize=(8, 4))
    for h, sub in per_cell.groupby("horizon"):
        ax.scatter([h] * len(sub), sub["amp_ratio"], label=f"h={h}", alpha=0.6)
    ax.axhline(1.0, color="black", lw=0.6, ls="--")
    ax.set_xlabel("horizon")
    ax.set_ylabel("median |pred|/|true|")
    ax.set_yscale("log")
    ax.set_title("Amplitude calibration (target = 1.0)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def plot_coverage(per_cell, out_path, target=0.80):
    _require_mpl()
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="coverage")
    fig, ax = plt.subplots(figsize=(1.2 * len(pivot.columns) + 2, 0.4 * len(pivot.index) + 2))
    im = ax.imshow(pivot.values, cmap="coolwarm", vmin=0.60, vmax=1.0, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label=f"q-coverage (target={target})")
    ax.set_title("Quantile coverage (q10–q90)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def plot_forecast_examples(pred_df, ret_panel, price_panel, tickers, quantiles, out_path, n_examples=6):
    _require_mpl()
    median_q = str(quantiles[len(quantiles) // 2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    windows = pred_df["window"].unique()
    if len(windows) == 0:
        return
    pick = np.linspace(0, len(windows) - 1, min(n_examples, len(windows))).astype(int)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, w in zip(axes.flat, [windows[i] for i in pick]):
        slab = pred_df[pred_df["window"] == w]
        tic = tickers[0]
        s = slab[slab["id"] == tic].sort_values("timestamp")
        if s.empty:
            continue
        fut_idx = pd.DatetimeIndex(s["timestamp"].values)
        last_p = price_panel[tic].iloc[price_panel.index.get_loc(fut_idx[0]) - 1]
        p_med = last_p * np.exp(np.cumsum(s[median_q].values))
        p_lo = last_p * np.exp(np.cumsum(s[lo_q].values))
        p_hi = last_p * np.exp(np.cumsum(s[hi_q].values))
        truth = price_panel[tic].loc[fut_idx]
        ax.plot(fut_idx, truth.values, "g.-", label="actual")
        ax.plot(fut_idx, p_med, "r.-", label="median")
        ax.fill_between(fut_idx, p_lo, p_hi, color="red", alpha=0.15)
        ax.set_title(f"{tic}  window={w}")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()

## 2. Section A — Path B config helpers (impl prompt step A)

In [ ]:
def normalize_path_b_config(cfg: dict) -> dict:
    """Ensure cfg["path_b"] exists with all Path-B knobs, derived from Path A fields.

    Sets (per path_b_implementation_prompt.md step A + B_plan_v2 §6.4):
      enabled, intervals_to_test, prediction_length=3, horizon=3,
      eval_horizons=[1,2,3], primary_horizons=[1,2,3], cross_learning_main=True,
      cross_learning_diagnostics=False, num_val_windows=3, refit_every_n_windows=1,
      val_step_size=cfg["walk_forward"]["shift"], output_root, scratchpad_root,
      current_state_path, quantiles, seed.
    Mutates and returns cfg.
    """
    pb = dict(cfg.get("path_b", {}) or {})

    pb.setdefault("enabled", True)

    # intervals_to_test: explicit list, else the single interval this config carries.
    if "intervals_to_test" not in pb:
        pb["intervals_to_test"] = [cfg["interval"]]
    pb["intervals_to_test"] = [int(x) for x in pb["intervals_to_test"]]

    # Path B horizons are fixed at 3 regardless of Path A's horizon=5.
    pb["prediction_length"] = int(pb.get("prediction_length", 3))
    pb["horizon"] = pb["prediction_length"]
    pb.setdefault("eval_horizons", [1, 2, 3])
    pb.setdefault("primary_horizons", [1, 2, 3])

    pb.setdefault("cross_learning_main", True)
    pb.setdefault("cross_learning_diagnostics", False)

    pb.setdefault("num_val_windows", 3)
    pb.setdefault("refit_every_n_windows", 1)
    pb.setdefault("val_step_size", cfg["walk_forward"]["shift"])

    # Output roots — default under path_b/ (user constraint: path_b folder only).
    pb.setdefault("output_root", "runs/path_b")
    pb.setdefault("scratchpad_root", "scratchpads/path_b")
    pb.setdefault("current_state_path", "current_state.md")

    # Quantiles inherited from Path A unless overridden.
    pb.setdefault("quantiles", cfg.get("quantiles", [0.1, 0.5, 0.9]))

    pb.setdefault("seed", cfg.get("seed", 42))

    # model_path: pinned for a clean A<->B comparison (verified-API note). amazon/chronos-2
    # matches Path A's HF weights; autogluon/chronos-2 is the same 120M model, AG mirror.
    pb.setdefault("model_path", "amazon/chronos-2")

    # AutoGluon fit defaults (inference batch is a separate knob from fine_tune batch).
    pb.setdefault("chronos2_batch_size", 64)
    pb.setdefault("enable_ensemble", False)
    pb.setdefault("time_limit", None)

    cfg["path_b"] = pb
    # Surface the Path B horizon to the top-level so build_chronos_inputs/anchors use it.
    cfg["pb_horizon"] = pb["prediction_length"]
    return cfg


def get_cross_learning_values(cfg: dict):
    """Return the cross_learning arm list: [True] normally; [True, False] if diagnostics on.

    Follows B_plan_v2 §3 exactly: main_value=cross_learning_main; if diagnostics,
    also run not(main_value).
    """
    pb = cfg.get("path_b", {})
    main_value = bool(pb.get("cross_learning_main", True))
    diagnostics = bool(pb.get("cross_learning_diagnostics", False))
    if diagnostics:
        return [main_value, not main_value]
    return [main_value]

## 3. Section B — AutoGluon data adapters (impl prompt step B)

In [ ]:
# Covariate split (impl prompt step B, "safe existing split"):
#   - calendar covariates  -> known-future  (leakage-safe; matches Path A leakage rule)
#   - market/index/futures/volume covariates -> past-only
# Recorded in current_state.md.

def split_covariates(cov_panel, interval):
    """Return (known_future_names, past_only_names) given the assembled cov panel.

    Calendar features are generated per-window (not in cov_panel) and are the only
    known-future covariates. Every column present in cov_panel (index/futures returns,
    dlog-volume) is past-only.
    """
    known_future = calendar_feature_names(interval)
    past_only = list(cov_panel.columns) if cov_panel is not None and not cov_panel.empty else []
    return known_future, past_only


def build_ag_train_frame(ret_panel, cov_panel, tickers, interval, covariate_mode="full",
                         context_window=None):
    """Build a single long-form AutoGluon train frame over the whole panel.

    Columns: item_id, timestamp, target, <past covariates>, <calendar known-future>.
    AutoGluon treats columns in `known_covariates_names` (the calendar set) as
    known-future and all other non-target/non-id columns as past covariates.

    context_window: optional (start_ts, end_ts) to restrict the training history.
    Returns (long_df, known_future_names, past_only_names).
    """
    use_market_cov = covariate_mode in ("full", "market_only") and cov_panel is not None and not cov_panel.empty
    use_calendar = covariate_mode in ("full", "calendar_only")

    idx = ret_panel.index
    if context_window is not None:
        start_ts, end_ts = context_window
        idx = idx[(idx >= start_ts) & (idx <= end_ts)]

    cal = calendar_features(idx, interval) if use_calendar else None
    cov = cov_panel.loc[idx] if use_market_cov else None

    blocks = []
    for tic in tickers:
        block = pd.DataFrame({"item_id": tic, "timestamp": idx, "target": ret_panel[tic].loc[idx].values})
        if use_market_cov:
            for c in cov.columns:
                block[c] = cov[c].values
        if use_calendar:
            for c in cal.columns:
                block[c] = cal[c].values
        blocks.append(block)
    long_df = pd.concat(blocks, ignore_index=True)

    known_future = calendar_feature_names(interval) if use_calendar else []
    past_only = list(cov.columns) if use_market_cov else []
    return long_df, known_future, past_only


def to_timeseries_dataframe(long_df, TimeSeriesDataFrame):
    """Wrap a long-form df (item_id, timestamp, target, covariates) as a
    TimeSeriesDataFrame. `TimeSeriesDataFrame` is injected so this module does not
    hard-depend on autogluon at import time."""
    return TimeSeriesDataFrame.from_data_frame(
        long_df, id_column="item_id", timestamp_column="timestamp"
    )


def build_known_covariates_for_window(fut_idx, tickers, interval, known_future_names):
    """Build the known-future covariate frame (calendar only) for one forecast horizon,
    in AutoGluon long form. Returns a df with item_id, timestamp, and the calendar cols."""
    if not known_future_names:
        return None
    cal = calendar_features(fut_idx, interval)
    blocks = []
    for tic in tickers:
        fb = pd.DataFrame({"item_id": tic, "timestamp": fut_idx})
        for c in known_future_names:
            fb[c] = cal[c].values
        blocks.append(fb)
    return pd.concat(blocks, ignore_index=True)

## 4. Section C — Path B run-directory helpers + append-only path_b_runs.csv (step C)

In [ ]:
PATH_B_RUNS_COLUMNS = [
    "run_id", "step_id", "parent_run_id", "seed", "train_universe_id", "eval_universe_id",
    "interval", "context_length", "prediction_length", "fine_tune", "fine_tune_mode",
    "fine_tune_lr", "fine_tune_steps", "fine_tune_batch_size", "batch_size",
    "cross_learning", "num_val_windows", "n_items_train", "n_rows_train",
    "fit_seconds", "predict_seconds", "primary_score", "primary_score_b0", "delta_vs_b0",
    "score_h1", "score_h1_b0", "delta_h1_vs_b0",
    "score_h2", "score_h2_b0", "delta_h2_vs_b0",
    "score_h3", "score_h3_b0", "delta_h3_vs_b0",
    "coverage", "coverage_b0", "delta_coverage_vs_b0",
    "overfit_flag", "runtime_flag", "selected_candidate", "notes",
]


def path_b_run_dir(output_root, interval, step_id, run_id):
    """Immutable run dir: runs/path_b/{interval}/{step_id}/{run_id}/ (impl prompt step C)."""
    d = Path(output_root) / f"{interval}m" / step_id / run_id
    d.mkdir(parents=True, exist_ok=True)
    (d / "preds").mkdir(exist_ok=True)
    (d / "logs").mkdir(exist_ok=True)
    (d / "plots").mkdir(exist_ok=True)
    return d


def runs_csv_path(output_root):
    return Path(output_root) / "path_b_runs.csv"


def append_path_b_run_row(output_root, row: dict):
    """Append one row to the append-only runs csv, creating it with the canonical header
    on first write. Missing columns are filled with NaN; never overwrites prior rows."""
    path = runs_csv_path(output_root)
    path.parent.mkdir(parents=True, exist_ok=True)
    full = {c: row.get(c, np.nan) for c in PATH_B_RUNS_COLUMNS}
    df = pd.DataFrame([full], columns=PATH_B_RUNS_COLUMNS)
    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, mode="w", header=True, index=False)


def load_hyperparameters(output_root, interval, step_id, run_id):
    """Load the exact hyperparameters dict saved for a run."""
    d = path_b_run_dir(output_root, interval, step_id, run_id)
    with open(d / "hyperparameters.json", "r", encoding="utf-8") as f:
        return json.load(f)

## 5. Section D — B0-B4 hyperparameter builders (impl prompt step D)

In [ ]:
# CHRONOS2_DEFAULTS must NOT contain cross_learning (B_plan_v2 §3): the flag is inserted
# explicitly in every builder so it is visible in hyperparameters + run metadata.
def chronos2_defaults(cfg):
    return {
        "model_path": cfg["path_b"]["model_path"],
        "batch_size": cfg["path_b"]["chronos2_batch_size"],
    }


def make_b0_hyperparameters(cfg, *, cross_learning: bool):
    """B0 — AutoGluon zero-shot reproduction (fine_tune=False)."""
    return {
        "Chronos2": {
            "fine_tune": False,
            "context_length": cfg["context_len"],
            "cross_learning": cross_learning,
            **chronos2_defaults(cfg),
            "ag_args": {"name_suffix": f"ZeroShot_cross{cross_learning}"},
        }
    }


def make_b1_hyperparameters(cfg, *, cross_learning: bool):
    """B1 — minimal LoRA smoke (lr 1e-5, steps 300, ft batch 8)."""
    return {
        "Chronos2": {
            "fine_tune": True,
            "fine_tune_mode": "lora",
            "fine_tune_lr": 1e-5,
            "fine_tune_steps": 300,
            "fine_tune_batch_size": 8,
            "context_length": cfg["context_len"],
            "fine_tune_context_length": min(cfg["context_len"], 2048),
            "cross_learning": cross_learning,
            **chronos2_defaults(cfg),
            "ag_args": {"name_suffix": f"LoRA_Min_cross{cross_learning}"},
        }
    }


def make_b2_hyperparameters(cfg, *, cross_learning: bool):
    """B2 — same config as B1 (repeatability across seeds)."""
    return make_b1_hyperparameters(cfg, cross_learning=cross_learning)


def make_b3_hyperparameters(cfg, *, steps, lr, fine_tune_batch_size, cross_learning: bool):
    """B3 — LoRA schedule sweep (steps / lr / ft batch size)."""
    return {
        "Chronos2": {
            "fine_tune": True,
            "fine_tune_mode": "lora",
            "fine_tune_lr": lr,
            "fine_tune_steps": steps,
            "fine_tune_batch_size": fine_tune_batch_size,
            "context_length": cfg["context_len"],
            "fine_tune_context_length": min(cfg["context_len"], 2048),
            "cross_learning": cross_learning,
            **chronos2_defaults(cfg),
            "ag_args": {
                "name_suffix": (
                    f"LoRA_steps{steps}_lr{lr}_ftbs{fine_tune_batch_size}_cross{cross_learning}"
                )
            },
        }
    }


def make_b4_hyperparameters(cfg, reference_lora_config, *, cross_learning: bool, batch_size: int):
    """B4 — cross-learning & inference batch-size stability around the B3 reference LoRA.

    Takes a deep copy of REFERENCE_LORA_CONFIG and overrides only cross_learning,
    inference batch_size, and the name suffix. Reference config holds exactly one
    fine-tuned candidate (no zero-shot arm)."""
    hp = copy.deepcopy(reference_lora_config)
    hp["Chronos2"]["cross_learning"] = cross_learning
    hp["Chronos2"]["batch_size"] = batch_size
    hp["Chronos2"]["ag_args"] = {"name_suffix": f"LoRA_Ref_cross{cross_learning}_batch{batch_size}"}
    return hp


# B3 staged grid (B_plan_v2 §B3): step sweep first, then LR around the best, then ft-batch.
@dataclass
class B3Params:
    steps: int
    lr: float
    fine_tune_batch_size: int


def b3_staged_grid():
    """Staged grid for compute control (B_plan_v2 §B3): not a blind full cross-product.
    Stage 1: fix lr=1e-5, ftbs=8, sweep steps.
    Stage 2: lr variants around the middle step count.
    Stage 3: ft-batch variant at the middle step count.
    """
    grid = []
    for steps in (100, 300, 1000):           # stage 1 — step sweep
        grid.append(B3Params(steps=steps, lr=1e-5, fine_tune_batch_size=8))
    grid.append(B3Params(steps=300, lr=3e-5, fine_tune_batch_size=8))   # stage 2 — lr variant
    grid.append(B3Params(steps=300, lr=1e-5, fine_tune_batch_size=16))  # stage 3 — ft-batch variant
    return grid


B4_GRID = [
    {"batch_size": 32, "role": "main_stability"},
    {"batch_size": 64, "role": "main_stability"},
    {"batch_size": 128, "role": "main_stability"},
]

## 6. Section E — Path B experiment runner + orchestrator (impl prompt step E)

In [ ]:
def _hp_chronos(hyperparameters):
    """Pull the single Chronos2 sub-dict out of a hyperparameters dict for metadata."""
    return hyperparameters.get("Chronos2", {})


def run_path_b_experiment(*, step_id, run_id, cfg, interval, ret_panel, cov_panel,
                          price_panel, eval_anchors, train_window, hyperparameters,
                          random_seed, ag_modules, b0_per_cell=None, notes="",
                          parent_run_id=None):
    """Fit one AutoGluon Chronos-2 candidate and evaluate it on the SAME walk-forward
    windows as Path A, writing predictions in Path A's schema so per_cell_metrics reuses
    the exact same code path.

    ag_modules: dict with the injected AutoGluon classes:
        {"TimeSeriesPredictor": ..., "TimeSeriesDataFrame": ...}
      (injected so this module imports without autogluon installed.)

    Returns a result dict with the run dir, per_cell metrics, summary, and timing.
    """
    TimeSeriesPredictor = ag_modules["TimeSeriesPredictor"]
    TimeSeriesDataFrame = ag_modules["TimeSeriesDataFrame"]

    pb = cfg["path_b"]
    horizon = pb["prediction_length"]
    quantiles = pb["quantiles"]
    tickers = cfg["tickers"]
    cov_mode = cfg["covariates"]

    run_dir = path_b_run_dir(pb["output_root"], interval, step_id, run_id)
    save_config_snapshot(cfg, str(run_dir))
    with open(run_dir / "hyperparameters.json", "w", encoding="utf-8") as f:
        json.dump(hyperparameters, f, indent=2, default=str)

    # ---- build the AutoGluon train frame (restricted to the train window) ----
    train_long, known_future, past_only = build_ag_train_frame(
        ret_panel, cov_panel, tickers, interval, cov_mode, context_window=train_window
    )
    n_items_train = train_long["item_id"].nunique()
    n_rows_train = len(train_long)
    train_tsdf = to_timeseries_dataframe(train_long, TimeSeriesDataFrame)

    # ---- fit ----
    predictor = TimeSeriesPredictor(
        target="target",
        prediction_length=horizon,
        known_covariates_names=known_future or None,
        quantile_levels=list(quantiles),
        eval_metric=pb.get("eval_metric", "WQL"),
        path=str(run_dir / "predictor"),
        verbosity=1,
    )
    fit_kwargs = dict(
        hyperparameters=hyperparameters,
        enable_ensemble=pb["enable_ensemble"],
        num_val_windows=pb["num_val_windows"],
        val_step_size=pb["val_step_size"],
        refit_every_n_windows=pb["refit_every_n_windows"],
        random_seed=random_seed,
    )
    if pb["time_limit"] is not None:
        fit_kwargs["time_limit"] = pb["time_limit"]

    t0 = time.time()
    predictor.fit(train_tsdf, **fit_kwargs)
    fit_seconds = time.time() - t0

    # ---- predict on the SAME eval anchors as Path A (Chronos predict_df schema) ----
    # For each anchor we feed context up to t_anchor-1 and known-future covariates over
    # the horizon, then collect quantile predictions in long form with columns
    # id, timestamp, "0.1", "0.5", "0.9", window, t_anchor — Path A's exact schema.
    all_preds = []
    t1 = time.time()
    for w, t_anchor in enumerate(tqdm(eval_anchors, desc=f"{step_id}/{run_id} eval")):
        pos = ret_panel.index.get_loc(t_anchor)
        ctx_start = ret_panel.index[max(0, pos - cfg["context_len"])]
        ctx_end = ret_panel.index[pos - 1]
        fut_idx = ret_panel.index[pos:pos + horizon]

        ctx_long, _, _ = build_ag_train_frame(
            ret_panel, cov_panel, tickers, interval, cov_mode,
            context_window=(ctx_start, ctx_end),
        )
        ctx_tsdf = to_timeseries_dataframe(ctx_long, TimeSeriesDataFrame)
        known_df = build_known_covariates_for_window(fut_idx, tickers, interval, known_future)
        known_tsdf = (
            to_timeseries_dataframe(known_df, TimeSeriesDataFrame) if known_df is not None else None
        )
        try:
            fc = predictor.predict(ctx_tsdf, known_covariates=known_tsdf, random_seed=random_seed)
        except Exception as e:  # noqa: BLE001 - record and continue, like Path A
            print(f"  window {w} ({t_anchor}) failed: {e}")
            continue
        pred_long = _ag_forecast_to_pathA_schema(fc, quantiles, tickers, fut_idx, w, t_anchor)
        all_preds.append(pred_long)
    predict_seconds = time.time() - t1

    preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    preds.to_parquet(run_dir / "preds" / "preds.parquet")

    # ---- metrics (reuse Path A per_cell_metrics + aggregate_metrics) ----
    per_cell = per_cell_metrics(preds, ret_panel, tickers, pb["eval_horizons"], quantiles)
    per_cell.to_csv(run_dir / "metrics.csv", index=False)
    agg = aggregate_metrics(per_cell)
    agg.to_csv(run_dir / "metrics_aggregate.csv", index=False)

    summary = _summarize(per_cell, pb["primary_horizons"], preds, step_id, run_id,
                         interval, cfg["context_len"], horizon, hyperparameters,
                         fit_seconds, predict_seconds, n_items_train, n_rows_train)
    with open(run_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=str)

    # ---- per-candidate Path A plots ----
    _render_path_a_plots(run_dir / "plots", per_cell, preds, ret_panel, price_panel,
                         tickers, pb["eval_horizons"], quantiles, title=f"{step_id}/{run_id}")

    # ---- append the run row (deltas vs B0 filled if b0_per_cell provided) ----
    row = _build_run_row(
        step_id=step_id, run_id=run_id, parent_run_id=parent_run_id, seed=random_seed,
        cfg=cfg, interval=interval, hyperparameters=hyperparameters,
        per_cell=per_cell, b0_per_cell=b0_per_cell, fit_seconds=fit_seconds,
        predict_seconds=predict_seconds, n_items_train=n_items_train,
        n_rows_train=n_rows_train, notes=notes,
    )
    append_path_b_run_row(pb["output_root"], row)

    return dict(run_dir=str(run_dir), per_cell=per_cell, aggregate=agg, summary=summary,
                preds=preds, run_row=row)


def _ag_forecast_to_pathA_schema(fc, quantiles, tickers, fut_idx, window, t_anchor):
    """Convert an AutoGluon forecast (TimeSeriesDataFrame, index (item_id, timestamp),
    quantile columns named like "0.1","0.5","0.9") to Path A's long preds schema:
    columns id, timestamp, "0.1","0.5","0.9", window, t_anchor."""
    df = fc.reset_index() if hasattr(fc, "reset_index") else pd.DataFrame(fc)
    df = df.rename(columns={"item_id": "id"})
    qcols = [str(q) for q in quantiles]
    # AutoGluon names quantile columns as the float repr; ensure our exact strings exist.
    keep = ["id", "timestamp"]
    for q in qcols:
        if q not in df.columns:
            # tolerate "mean"/alt naming for the median; fall back to nearest column
            if q == str(quantiles[len(quantiles) // 2]) and "mean" in df.columns:
                df[q] = df["mean"]
            else:
                df[q] = np.nan
        keep.append(q)
    out = df[keep].copy()
    out["window"] = window
    out["t_anchor"] = t_anchor
    return out


def _summarize(per_cell, primary_horizons, preds, step_id, run_id, interval, context_len,
               horizon, hyperparameters, fit_seconds, predict_seconds, n_items_train, n_rows_train):
    chr_hp = _hp_chronos(hyperparameters)
    if per_cell is None or per_cell.empty:
        primary = pd.DataFrame()
    else:
        primary = per_cell[per_cell["horizon"].isin(primary_horizons)]
    return dict(
        step_id=step_id, run_id=run_id, interval=interval, context_length=context_len,
        prediction_length=horizon,
        fine_tune=bool(chr_hp.get("fine_tune", False)),
        fine_tune_mode=chr_hp.get("fine_tune_mode"),
        cross_learning=chr_hp.get("cross_learning"),
        batch_size=chr_hp.get("batch_size"),
        n_windows=int(preds["window"].nunique()) if not preds.empty else 0,
        n_items_train=n_items_train, n_rows_train=n_rows_train,
        mean_da_primary=float(primary["da"].mean()) if not primary.empty else None,
        median_da_primary=float(primary["da"].median()) if not primary.empty else None,
        cells_signif_05=int((primary["p_binom_bh"] < 0.05).sum()) if not primary.empty else 0,
        mean_pearson_primary=float(primary["pearson"].mean()) if not primary.empty else None,
        mean_coverage_primary=float(primary["coverage"].mean()) if not primary.empty else None,
        fit_seconds=fit_seconds, predict_seconds=predict_seconds,
    )


def _render_path_a_plots(plots_dir, per_cell, preds, ret_panel, price_panel, tickers,
                         horizons, quantiles, title=""):
    plots_dir = Path(plots_dir)
    plots_dir.mkdir(parents=True, exist_ok=True)
    if per_cell is None or per_cell.empty or preds.empty:
        return
    try:
        plot_da_heatmap(per_cell, plots_dir / "da_heatmap.png", title=title)
        plot_corr_hist(preds, ret_panel, horizons, quantiles, plots_dir / "corr_hist.png")
        plot_amplitude(per_cell, plots_dir / "amplitude.png")
        plot_coverage(per_cell, plots_dir / "coverage.png")
        plot_forecast_examples(preds, ret_panel, price_panel, tickers, quantiles,
                               plots_dir / "forecast_examples.png")
    except Exception as e:  # noqa: BLE001
        print(f"  plot render warning ({title}): {e}")


def _score_by_horizon(per_cell, h):
    """Primary per-horizon score = mean DA at horizon h (None if absent)."""
    if per_cell is None or per_cell.empty:
        return None
    sub = per_cell[per_cell["horizon"] == h]
    return float(sub["da"].mean()) if not sub.empty else None


def _primary_score(per_cell, primary_horizons):
    if per_cell is None or per_cell.empty:
        return None
    sub = per_cell[per_cell["horizon"].isin(primary_horizons)]
    return float(sub["da"].mean()) if not sub.empty else None


def _coverage_score(per_cell, primary_horizons):
    if per_cell is None or per_cell.empty:
        return None
    sub = per_cell[per_cell["horizon"].isin(primary_horizons)]
    return float(sub["coverage"].mean()) if not sub.empty else None


def _delta(a, b):
    if a is None or b is None:
        return None
    return a - b


def _build_run_row(*, step_id, run_id, parent_run_id, seed, cfg, interval, hyperparameters,
                   per_cell, b0_per_cell, fit_seconds, predict_seconds, n_items_train,
                   n_rows_train, notes):
    pb = cfg["path_b"]
    chr_hp = _hp_chronos(hyperparameters)
    ph = pb["primary_horizons"]

    primary = _primary_score(per_cell, ph)
    primary_b0 = _primary_score(b0_per_cell, ph) if b0_per_cell is not None else None
    cov = _coverage_score(per_cell, ph)
    cov_b0 = _coverage_score(b0_per_cell, ph) if b0_per_cell is not None else None

    sh = {h: _score_by_horizon(per_cell, h) for h in (1, 2, 3)}
    sh_b0 = {h: (_score_by_horizon(b0_per_cell, h) if b0_per_cell is not None else None) for h in (1, 2, 3)}

    return dict(
        run_id=run_id, step_id=step_id, parent_run_id=parent_run_id, seed=seed,
        train_universe_id="core12", eval_universe_id="core12",
        interval=interval, context_length=cfg["context_len"],
        prediction_length=pb["prediction_length"],
        fine_tune=bool(chr_hp.get("fine_tune", False)),
        fine_tune_mode=chr_hp.get("fine_tune_mode"),
        fine_tune_lr=chr_hp.get("fine_tune_lr"),
        fine_tune_steps=chr_hp.get("fine_tune_steps"),
        fine_tune_batch_size=chr_hp.get("fine_tune_batch_size"),
        batch_size=chr_hp.get("batch_size"),
        cross_learning=chr_hp.get("cross_learning"),
        num_val_windows=pb["num_val_windows"],
        n_items_train=n_items_train, n_rows_train=n_rows_train,
        fit_seconds=fit_seconds, predict_seconds=predict_seconds,
        primary_score=primary, primary_score_b0=primary_b0, delta_vs_b0=_delta(primary, primary_b0),
        score_h1=sh[1], score_h1_b0=sh_b0[1], delta_h1_vs_b0=_delta(sh[1], sh_b0[1]),
        score_h2=sh[2], score_h2_b0=sh_b0[2], delta_h2_vs_b0=_delta(sh[2], sh_b0[2]),
        score_h3=sh[3], score_h3_b0=sh_b0[3], delta_h3_vs_b0=_delta(sh[3], sh_b0[3]),
        coverage=cov, coverage_b0=cov_b0, delta_coverage_vs_b0=_delta(cov, cov_b0),
        overfit_flag=False, runtime_flag=False, selected_candidate=False, notes=notes,
    )


def run_path_b(cfg_path: str, steps=("b0", "b1", "b2", "b3", "b4"),
               panels=None, ag_modules=None, cache_dir=None):
    """Top-level Path B orchestrator (impl prompt step E).

    Order: B0 first (harness ref) -> B1 -> B2 -> B3 -> select REFERENCE_LORA_CONFIG
    from the written B3 table (NOT inside the training loop) -> B4 -> aggregate compare.

    panels: optional pre-built (price_panel, ret_panel, cov_panel). If None, the caller
            (runner_path_b.ipynb) must build them via Path A loaders (`%run basic_cells`)
            and pass them in — this module does not do ISS/Drive IO. See current_state.md.
    ag_modules: {"TimeSeriesPredictor":..., "TimeSeriesDataFrame":...}. Required.
    """
    if ag_modules is None:
        raise ValueError(
            "ag_modules is required: pass {'TimeSeriesPredictor':..., 'TimeSeriesDataFrame':...} "
            "from autogluon.timeseries. The runner injects these."
        )
    # steps selector: accept a tuple/list/set of step names from {"b0".."b4"}.
    # Reject a bare string ("b0") — the missing-comma trap iterates characters.
    if isinstance(steps, str):
        raise TypeError(
            f"steps must be a tuple/list of step names, e.g. (\'b0\',) or "
            f"(\'b0\',\'b1\') — got the bare string {steps!r}."
        )
    steps = tuple(steps)
    cfg = normalize_path_b_config(load_config(cfg_path))
    pb = cfg["path_b"]
    interval = cfg["interval"]
    Path(pb["output_root"]).mkdir(parents=True, exist_ok=True)

    if panels is None:
        raise ValueError(
            "panels=None: Path B does not load ISS/Drive data itself. The runner must "
            "build (price_panel, ret_panel, cov_panel) via Path A loaders and pass them in. "
            "See path_b/current_state.md (TODO/ASK: data handoff)."
        )
    price_panel, ret_panel, cov_panel = panels

    # Eval anchors — same walk-forward scheme as Path A, but Path B horizon=3.
    wf = cfg["walk_forward"]
    eval_anchors = walk_forward_anchors(
        ret_panel.index, cfg["context_len"], pb["prediction_length"],
        shift=wf["shift"], max_windows=wf.get("max_windows"),
    )
    print(f"Path B: interval={interval}m  context_len={cfg['context_len']}  "
          f"H={pb['prediction_length']}  eval windows={len(eval_anchors)}")

    # Train window = history strictly BEFORE the first eval anchor (no leakage into eval).
    first_anchor = eval_anchors[0] if eval_anchors else ret_panel.index[cfg["context_len"]]
    train_window = (ret_panel.index[0], ret_panel.index[ret_panel.index.get_loc(first_anchor) - 1])

    seed = pb["seed"]
    results = {"b0": [], "b1": [], "b2": [], "b3": [], "b4": []}

    def _run(step_id, run_id, hp, rseed, b0_pc, parent=None, notes=""):
        return run_path_b_experiment(
            step_id=step_id, run_id=run_id, cfg=cfg, interval=interval,
            ret_panel=ret_panel, cov_panel=cov_panel, price_panel=price_panel,
            eval_anchors=eval_anchors, train_window=train_window, hyperparameters=hp,
            random_seed=rseed, ag_modules=ag_modules, b0_per_cell=b0_pc,
            parent_run_id=parent, notes=notes,
        )

    # ---- B0 (always run; it is the comparison anchor for every fine-tuned candidate) ----
    b0_per_cell = None
    if "b0" in steps:
        for cl in get_cross_learning_values(cfg):
            r = _run("b0_autogluon_zeroshot", f"seed_main_cross_{cl}",
                     make_b0_hyperparameters(cfg, cross_learning=cl), seed, None,
                     notes="B0 zero-shot harness reference")
            results["b0"].append(r)
        # B0 reference = the main-arm (cross_learning_main) zero-shot per_cell.
        main_cl = bool(pb["cross_learning_main"])
        for r in results["b0"]:
            if r["run_row"]["cross_learning"] == main_cl:
                b0_per_cell = r["per_cell"]
                break
        if b0_per_cell is None and results["b0"]:
            b0_per_cell = results["b0"][0]["per_cell"]

    # ---- B1 minimal LoRA ----
    if "b1" in steps:
        for cl in get_cross_learning_values(cfg):
            results["b1"].append(_run(
                "b1_lora_min", f"seed_main_cross_{cl}",
                make_b1_hyperparameters(cfg, cross_learning=cl), seed, b0_per_cell,
                parent="b0_autogluon_zeroshot", notes="B1 minimal LoRA",
            ))

    # ---- B2 repeatability (seeds 1,2,3) ----
    if "b2" in steps:
        for cl in get_cross_learning_values(cfg):
            for s in (1, 2, 3):
                results["b2"].append(_run(
                    "b2_lora_repeatability", f"seed_{s}_cross_{cl}",
                    make_b2_hyperparameters(cfg, cross_learning=cl), s, b0_per_cell,
                    parent="b1_lora_min", notes="B2 repeatability",
                ))

    # ---- B3 schedule sweep ----
    if "b3" in steps:
        for p in b3_staged_grid():
            for cl in get_cross_learning_values(cfg):
                results["b3"].append(_run(
                    "b3_lora_schedule",
                    f"steps_{p.steps}_lr_{p.lr}_ftbs_{p.fine_tune_batch_size}_cross_{cl}",
                    make_b3_hyperparameters(cfg, steps=p.steps, lr=p.lr,
                                            fine_tune_batch_size=p.fine_tune_batch_size,
                                            cross_learning=cl),
                    seed, b0_per_cell, parent="b1_lora_min", notes="B3 schedule sweep",
                ))

    # ---- Select REFERENCE_LORA_CONFIG from the WRITTEN B3 table (not in the loop) ----
    reference_lora_config = None
    if "b4" in steps:
        b3_table = collect_path_b_runs(pb["output_root"], step_id="b3_lora_schedule")
        main_cl = bool(pb["cross_learning_main"])
        ref_run_id = choose_reference_candidate(
            b3_table[b3_table["cross_learning"] == main_cl],
            primary_key="delta_vs_b0",
            stability_keys=["delta_h1_vs_b0", "delta_h2_vs_b0", "delta_h3_vs_b0", "fit_seconds"],
        )
        if ref_run_id is not None:
            reference_lora_config = load_hyperparameters(
                pb["output_root"], interval, "b3_lora_schedule", ref_run_id
            )
        elif "b1" in steps:
            # Fallback: if B3 wasn't run / empty, use the B1 main-arm config as reference.
            reference_lora_config = make_b1_hyperparameters(cfg, cross_learning=main_cl)

    # ---- B4 cross-learning & batch-size stability around the reference LoRA ----
    if "b4" in steps and reference_lora_config is not None:
        for cl in get_cross_learning_values(cfg):
            for p in B4_GRID:
                results["b4"].append(_run(
                    "b4_cross_learning_stability",
                    f"cross_{cl}_batch_{p['batch_size']}_{p['role']}",
                    make_b4_hyperparameters(cfg, reference_lora_config,
                                            cross_learning=cl, batch_size=p["batch_size"]),
                    seed, b0_per_cell, parent="b3_lora_schedule", notes="B4 stability",
                ))

    # ---- Aggregate comparison across all written runs ----
    comparison = compare_path_b_to_b0(pb["output_root"])
    plots_root = Path(pb["output_root"]) / f"{interval}m" / "_comparison_plots"
    make_comparison_plots(comparison, plots_root)

    return dict(cfg=cfg, results=results, comparison=comparison,
                reference_lora_config=reference_lora_config,
                runs_csv=str(runs_csv_path(pb["output_root"])),
                comparison_plots_dir=str(plots_root))

## 7. Section F — Comparison utilities + comparison plots (impl prompt step F)

In [ ]:
def collect_path_b_runs(output_root, step_id=None):
    """Read the append-only runs csv; optionally filter to one step_id."""
    path = runs_csv_path(output_root)
    if not path.exists():
        return pd.DataFrame(columns=PATH_B_RUNS_COLUMNS)
    df = pd.read_csv(path)
    if step_id is not None:
        df = df[df["step_id"] == step_id].copy()
    return df


def choose_reference_candidate(runs, primary_key="delta_vs_b0",
                               stability_keys=None, preference="simplest_with_good_score"):
    """Pick the B3 reference run AFTER all candidates are written (B_plan_v2 §B3).

    Strategy: among runs whose primary_key is at/near the top, prefer the simplest
    schedule (fewest fine_tune_steps) that stays within tolerance of the best score.
    Returns the run_id (str) or None if the table is empty.
    """
    if runs is None or len(runs) == 0:
        return None
    runs = runs.copy()
    runs = runs[~runs[primary_key].isna()]
    if len(runs) == 0:
        # No deltas computed (e.g. B0 absent) — fall back to raw primary_score.
        runs = collect_path_b_runs_fallback(runs)
        if len(runs) == 0:
            return None
    best = runs[primary_key].max()
    # tolerance band: within 1 standard-deviation of seed noise if available, else 0.5pp
    tol = 0.005
    near_best = runs[runs[primary_key] >= best - tol]
    if preference == "simplest_with_good_score" and "fine_tune_steps" in near_best.columns:
        near_best = near_best.sort_values(["fine_tune_steps", primary_key],
                                          ascending=[True, False])
    else:
        near_best = near_best.sort_values(primary_key, ascending=False)
    return str(near_best.iloc[0]["run_id"])


def collect_path_b_runs_fallback(runs):
    """If delta columns are all NaN, fall back to raw primary_score ranking."""
    runs = runs.copy()
    if "primary_score" in runs.columns:
        runs = runs[~runs["primary_score"].isna()]
        runs = runs.rename(columns={"primary_score": "delta_vs_b0"})
    return runs


def compare_path_b_to_b0(output_root):
    """Build the full comparison table: every run with its delta-vs-B0 columns.
    B0 rows have delta 0 by construction. Returns the runs DataFrame sorted by step/run."""
    df = collect_path_b_runs(output_root)
    if df.empty:
        return df
    return df.sort_values(["interval", "step_id", "run_id"]).reset_index(drop=True)


def make_comparison_plots(comparison, plots_dir):
    """Comparison plots (B_plan_v2 §5.4 / impl prompt step F):
    delta_by_run, delta_by_horizon, delta_by_ticker_heatmap (placeholder—needs per-cell),
    runtime_vs_delta, coverage_candidate_vs_b0, cross_learning_batch_stability."""
    if plt is None or comparison is None or comparison.empty:
        return
    plots_dir = Path(plots_dir)
    plots_dir.mkdir(parents=True, exist_ok=True)
    ft = comparison[comparison["fine_tune"] == True].copy()  # noqa: E712

    # delta_by_run
    if not ft.empty and ft["delta_vs_b0"].notna().any():
        fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(ft)), 4))
        ax.bar(ft["run_id"].astype(str), ft["delta_vs_b0"].fillna(0.0))
        ax.axhline(0, color="black", lw=0.7)
        ax.set_ylabel("primary DA − B0")
        ax.set_title("Δ vs B0 per fine-tuned run")
        ax.tick_params(axis="x", rotation=85, labelsize=6)
        plt.tight_layout(); plt.savefig(plots_dir / "delta_by_run.png", dpi=120); plt.close()

    # delta_by_horizon
    if not ft.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        for _, r in ft.iterrows():
            ys = [r.get("delta_h1_vs_b0"), r.get("delta_h2_vs_b0"), r.get("delta_h3_vs_b0")]
            ax.plot([1, 2, 3], ys, marker="o", alpha=0.5)
        ax.axhline(0, color="black", lw=0.7)
        ax.set_xticks([1, 2, 3]); ax.set_xlabel("horizon"); ax.set_ylabel("DA − B0")
        ax.set_title("Δ vs B0 by horizon (one line per run)")
        plt.tight_layout(); plt.savefig(plots_dir / "delta_by_horizon.png", dpi=120); plt.close()

    # runtime_vs_delta
    if not ft.empty and ft["delta_vs_b0"].notna().any():
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(ft["fit_seconds"], ft["delta_vs_b0"])
        ax.axhline(0, color="black", lw=0.7)
        ax.set_xlabel("fit_seconds"); ax.set_ylabel("primary DA − B0")
        ax.set_title("Runtime vs metric lift")
        plt.tight_layout(); plt.savefig(plots_dir / "runtime_vs_delta.png", dpi=120); plt.close()

    # coverage_candidate_vs_b0
    if not ft.empty and ft["coverage"].notna().any():
        fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(ft)), 4))
        ax.bar(ft["run_id"].astype(str), ft["delta_coverage_vs_b0"].fillna(0.0))
        ax.axhline(0, color="black", lw=0.7)
        ax.set_ylabel("coverage − B0 coverage")
        ax.set_title("Calibration vs B0")
        ax.tick_params(axis="x", rotation=85, labelsize=6)
        plt.tight_layout(); plt.savefig(plots_dir / "coverage_candidate_vs_b0.png", dpi=120); plt.close()

    # cross_learning_batch_stability (B4 only)
    b4 = comparison[comparison["step_id"] == "b4_cross_learning_stability"].copy()
    if not b4.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        for cl, sub in b4.groupby("cross_learning"):
            sub = sub.sort_values("batch_size")
            ax.plot(sub["batch_size"], sub["delta_vs_b0"], marker="o", label=f"cross={cl}")
        ax.axhline(0, color="black", lw=0.7)
        ax.set_xlabel("inference batch_size"); ax.set_ylabel("primary DA − B0")
        ax.set_title("B4 cross-learning × batch-size stability")
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.savefig(plots_dir / "cross_learning_batch_stability.png", dpi=120); plt.close()

## Usage in `runner_B.ipynb`

```python
%run basic_cells_A.ipynb   # Path A loaders (data, FORTS, cache)
%run basic_cells_B.ipynb   # this file — Path B functions land in globals

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
ag_modules = {'TimeSeriesPredictor': TimeSeriesPredictor, 'TimeSeriesDataFrame': TimeSeriesDataFrame}
summary = run_path_b(CONFIG_PATH, steps=('b0','b1','b2','b3','b4'), panels=panels, ag_modules=ag_modules)
```
